In [5]:
from langchain.tools import tool 
import requests
from bs4 import BeautifulSoup
from tavily import TavilyClient
import os 
from dotenv import load_dotenv
from rich import print
load_dotenv()

tavily = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))


In [6]:
''' this is the web scraping tool which gives the content of the page by cleaning the scraped data  '''
@tool
def scrape_url(url: str) -> str:
    """Scrape and return clean text content from a given URL for deeper reading."""
    try:
        resp = requests.get(url, timeout=8, headers={"User-Agent": "Mozilla/5.0"})
        soup = BeautifulSoup(resp.text, "html.parser")
        for tag in soup(["script", "style", "nav", "footer"]):
            tag.decompose()
        return soup.get_text(separator=" ", strip=True)
    except Exception as e:
        return f"Could not scrape URL: {str(e)}"



In [29]:

def web_search(query : str) -> str:
    """Search the web for recent and reliable information on a topic . Returns Titles , URLs and snippets."""
    results = tavily.search(query=query,max_results=5)

    out = []

    for r in results['results']:
        out.append(
            f"Title: {r['title']}\nURL: {r['url']}\nSnippet: {r['content'][:300]}\n"
        )
    
    return "\n----\n".join(out)


In [30]:
 results=web_search("what is a cat in animals")

In [31]:
print(results)

Title: Facts you need to know about domestic cats
URL: https://www.ifaw.org/animals/cats
Snippet: ##### What is a cat? Cats are one of the most popular pets on the planet, but unlike dogs, humans haven’t 
had a large hand in selectively breeding the species. Instead, we developed a relationship with cats over thousands
of years thanks to our need to stop rodents from eating our food and cats’ nat

----
Title: Cats & the Feline Family Portal | Britannica
URL: https://www.britannica.com/browse/Cats
Snippet: Cat, (Felis catus), also called house cat or domestic cat, domesticated member of the family Felidae, 
order Carnivora, and the smallest member of that family.

----
Title: Domestic cat | National Geographic
URL: https://www.nationalgeographic.com/animals/mammals/facts/domestic-cat
Snippet: # Domestic cat. From ancient Egyptians to today’s internet users, people have always loved their cats. 
There are at least 45 domestic breeds, which differ widely in features such as coat color, tail length, hair 
texture, and temperament, according to the Cat Fancier’s Association. One of the most un

----
Title: Cats | Healthy Pets, Healthy People
URL: https://www.cdc.gov/healthy-pets/about/cats.html
Snippet: A **.gov** website belongs to an official government organization in the United States. A **lock** (  ) or
**https://** means you've safely connected to the .gov website. # Cats. Although cats are great companions, cat 
owners should be aware that sometimes cats can carry harmful germs that can make 

----
Title: The Cat Family Explained: From Lions to Domestic Cats
URL: https://headsupfortails.com/blogs/cats/the-cat-family-lions-tigers-domestic-cats
Snippet: Discover the fascinating Cat Family, from roaring lions to your domestic cat. Learn traits, lineages, and 
fun facts about all felids!

In [21]:
urls=[]
for r in results:
        urls.append(r.findAll(url))
urls

AttributeError: 'str' object has no attribute 'findAll'

In [22]:
results = tavily.search(query="cats",max_results=5)
results

{'query': 'cats',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://en.wikipedia.org/wiki/Cat',
   'title': 'Cat - Wikipedia',
   'content': '# Cat. This is a good article. This article is about the species commonly kept as a pet. For the cat family, see Felidae. | Cat  Temporal range: 0.0095–0\xa0Ma PreꞒ  Ꞓ  O  S  D  C  P  T  J  K  Pg  N   Holocene to present (9,500 years ago) |. The **cat** (***Felis catus***), also called **domestic cat** and **house cat**, is a small domesticated carnivorous mammal. It is a member of Felidae, the family of mammals in the order "Order (taxonomy)") Carnivora also colloquially referred to as cats. The domestic cat is the only domesticated species of the family Felidae. Advances in archaeology and genetics have shown that the domestication of the cat started in the Near East around 7500 BCE. About 73.8\xa0million cats are estimated to live in the United States, and about 10.9\xa0million cats in the United Kingd

In [ ]:
import mysql.connector
from mysql.connector import Error
from langchain_core.tools import tool

@tool
def database_operation(query: str) -> str:
    """
    Executes a SQL query against the local MySQL database.
    Input should be a fully formed SQL string (e.g., SELECT, INSERT, UPDATE).
    Returns the query results or a success/error message.
    """
    connection = None
    cursor = None
    try:
        connection = mysql.connector.connect(
            host="localhost",
            user="root",
            password="your_password",
            database="your_database_name"
        )
        
        if connection.is_connected():
            cursor = connection.cursor(dictionary=True) # Returns rows as dictionaries
            cursor.execute(query)
            
            # Handle Read queries (SELECT)
            if query.strip().upper().startswith("SELECT"):
                results = cursor.fetchall()
                return str(results)
            
            # Handle Write queries (INSERT, UPDATE, DELETE)
            else:
                connection.commit()
                return f"Database operation successful. Affected rows: {cursor.rowcount}"
                
    except Error as e:
        # Return the error to the agent so it knows what went wrong
        return f"Database error encountered: {e}"
        
    finally:
        # Ensure connections are always closed safely
        if cursor:
            cursor.close()
        if connection and connection.is_connected():
            connection.close()
